# AISingers

Русский Colab-блокнот для [AISingers](https://github.com/varaslaw/ultimate-rvc): AI-каверы и TTS без вкладки обучения.

Нажмите `Runtime` -> `Run all`, дождитесь публичной ссылки в последней ячейке и откройте ее. Если стандартная ссылка Gradio работает медленно, выберите `ngrok` и вставьте личный токен с [ngrok.com](https://ngrok.com/).


In [ ]:
# @title 0: Инициализация блокнота
%pip install ipython-autotime pyngrok
%load_ext autotime

import os
import time
from pathlib import Path
from urllib import request

import ipywidgets as widgets
from IPython.display import clear_output
from IPython.display import display as i_display
from pyngrok import ngrok

clear_output()


In [ ]:
# @title 1: Клонируем AISingers
!git clone https://github.com/varaslaw/ultimate-rvc AISingers
%cd /content/AISingers
clear_output()


In [ ]:
# @title 2: Устанавливаем зависимости
prerelease = "--prerelease if-necessary-or-explicit"

!apt-get update -qq 2>&1 | grep -v "r2u.stat.illinois.edu"
!apt install -y python3-dev unzip 2>&1 | grep -v "is not a symbolic link"
!curl -LsSf https://astral.sh/uv/0.9.11/install.sh | sh

os.environ["URVC_CONSOLE_LOG_LEVEL"] = "WARNING"

!uv run -q $prerelease ./src/ultimate_rvc/core/main.py
!uv add $prerelease matplotlib-inline==0.1.7
clear_output()


In [ ]:
# @title 3: Запуск AISingers
# @markdown #### Способ публикации интерфейса:

method = "gradio"  # @param ["gradio", "ngrok", "cloudflared", "localtunnel"]
ngrok_token = ""  # @param {type:"string"}
run_path = "./src/ultimate_rvc/web/main.py"

if method == "gradio":
    !uv run $prerelease $run_path --share
elif method == "ngrok":
    try:
        ngrok.set_auth_token(ngrok_token)
        ngrok.kill()
        tunnel = ngrok.connect(6969)
        print(f"AISingers URL: {tunnel.public_url}")
        !uv run $prerelease $run_path --listen-port 6969
    except Exception as e:  # noqa: BLE001
        print(f"Не удалось запустить ngrok: {e}")
elif method == "cloudflared":
    !curl -LO https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb
    clear_output()
    !rm -rf nohup.out
    !nohup cloudflared tunnel --url localhost:6969 &
    clear_output()
    time.sleep(10)
    cloudflare_url = !grep -oE "https://[a-zA-Z0-9.-]+\.trycloudflare\.com" nohup.out
    print(f"AISingers URL: {cloudflare_url}")
    !uv run $prerelease $run_path --listen-port 6969
elif method == "localtunnel":
    !npm install -g localtunnel &>/dev/null
    Path("url.txt").open("w", encoding="utf-8").close()
    !lt --port 6969 >> url.txt 2>&1 &
    time.sleep(2)
    endpoint_ip = (
        request.urlopen("https://ipv4.icanhazip.com").read().decode("utf8").strip("\n")
    )
    tunnel_url = (
        Path("url.txt").read_text(encoding="utf-8").replace("your url is: ", "")
    )
    print(f"AISingers URL: {tunnel_url}")
    password_endpoint_widget = widgets.Text(
        value=endpoint_ip,
        description="IP-пароль:",
        disabled=True,
    )
    i_display(password_endpoint_widget)
    !uv run $prerelease $run_path --listen-port 6969
